In [ ]:
import pandas as pd
import os
import requests

logo_df = pd.read_csv("team_logos.csv")
print(logo_df.head())

for col, value in logo_df.iterrows():
    print(f'column: {col}: value {value}')

logo_unique = logo_df['team_name'].unique
print('team names', {logo_unique})


In [ ]:
def split_csv(line: str):
    # preserves empty fields between consecutive commas
    return [x.strip() for x in line.split(",")]

def is_blank(x: str) -> bool:
    return x is None or str(x).strip() == ""

def merge_two_rows(alias_row: str, canonical_row: str):
    a = split_csv(alias_row)      # e.g., LA_CAR
    b = split_csv(canonical_row)  # e.g., LAR_CAR

    if len(a) != len(b):
        raise ValueError(f"Row length mismatch: {len(a)} vs {len(b)}")

    merged = b[:]  # base = canonical

    # Force canonical identifiers (adjust if your game_id convention differs)
    merged[0] = "2025"            # season
    merged[1] = "19"              # week
    merged[2] = "2025_19_LAR_CAR" # game_id
    merged[3] = "2026-01-10"      # date
    merged[4] = "CAR"            # home
    merged[5] = "LAR"            # away

    # Backfill blanks from alias row
    for i in range(len(merged)):
        if is_blank(merged[i]) and not is_blank(a[i]):
            merged[i] = a[i]

    return ",".join(merged)

# ---- usage ----
alias = """2025,19,2025_19_LA_CAR,2026-01-10,CAR,LAR,,,,,12.0,21.5,0.0,-87.5,45.0,-1.75,43.0,8.0,6.5,56.0,6.5,-0.2218636215893796,0.4461538461538463,0.023076923076923,0.0472413793103448,0.4634835638851705,0.0427636471813178,0.0268795094874052,0.0183697753978878,16.0,20.75,0.25,-119.5,86.0,-2.25,43.0,10.0,6.75,57.66666666666666,7.0,-0.05321112651818,0.4522435897435897,0.0396138583638583,0.0275889436234263,0.4668608295616327,0.0362627759716113,-0.039143441676494,0.0121610781751343,30.5,23.5,0.5,210.0,-525.0,3.5,49.0,9.0,7.0,51.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.75,29.75,0.5,180.5,-354.25,3.5,48.75,8.0,6.25,51.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-18.5,-2.0,-0.5,-297.5,570.0,-5.25,-6.0,-1.0,-0.5,,,,,,,,,,,-18.75,-9.0,-0.25,-300.0,440.25,-5.75,-5.75,2.0,0.5,6.666666666666664,-10.0,,,,,,,,,WC,0.1904761904761904,0.8518518518518519,-0.6613756613756614,0.182741116751269,0.817258883248731,-0.6345177664974619,-10.5,45.5,7,6,1,,outdoors,grass,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,2026-01-10,5,True,True,False,True,202519,1385.443862821534,1611.1870721914413,-225.7432093699074,,,,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
"""
canon = """2025,19,2025_19_LAR_CAR,2026-01-10,CAR,LAR,,,,,15.666666666666666,21.0,0.3333333333333333,-110.0,73.33333333333333,-2.1666666666666665,43.5,8.666666666666666,6.666666666666667,57.66666666666666,7.0,-0.100144214656693,0.4641025641025642,0.0296703296703296,0.0314942528735632,0.4518461854472566,0.0443821139938944,-0.0194820753785462,0.0162147709001792,19.0,22.2,0.4,-210.6,153.8,-3.8,43.3,9.4,6.6,54.5,6.0,-0.0086167217561098,0.4517948717948717,0.0441910866910866,0.0220711548987411,0.4996425098031524,0.0443948361619044,-0.0680811048396861,0.0189596317708767,32.666666666666664,28.33333333333333,0.3333333333333333,174.0,-390.6666666666667,2.833333333333333,46.833333333333336,7.333333333333333,6.0,51.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.8,27.2,0.6,44.4,-207.4,0.9,48.9,7.8,6.4,51.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-17.0,-7.333333333333332,0.0,-284.0,464.0,-5.0,-3.333333333333336,1.333333333333333,0.666666666666667,6.666666666666664,-10.0,,,,,,,,,-17.799999999999997,-5.0,-0.1999999999999999,-255.0,361.2000000000001,-4.7,-5.600000000000001,1.6000000000000003,0.1999999999999993,3.5,-11.0,,,,,,,,,WC,,,,,,,,,,,,,,,,,2026-01-10T13:30:00-08:00,21,,,,,,,,,,,,,,,,,,,,,,,,,,,,2026-01-10,5,True,True,False,True,202519,1385.443862821534,1611.1870721914413,-225.7432093699074,,,,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
"""

print(merge_two_rows(alias, canon))
